# Variational Quantum Circuit

Raw OPM-MEG → Preprocessing → Epoching → Trial tensors → TT decomposition → TT features → Dimensionality reduction → Angle encoding → VQC → Classification

The purpose of the VQC is to determine whether the compressed representation of the MEG trial contains information that can distinguish the four tasks:
{auditory,somatosensory,motor,rest}.

1. Takes 32 MEG trials.
2. Runs them through the quantum circuit.
3. Gets four outputs.
4. Applies softmax.
5. Compares predictions against the true labels.
6. Calculates cross-entropy loss.
7. Uses Adam to update the VQC parameters.

Conceptually:
MEG features → VQC(θ) → prediction → loss → ∂θ/∂L → update θ.

In [9]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3.13 install --upgrade pip


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import seaborn as sns

import tensorly as tl
from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor
from pennylane import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_recall_fscore_support

import pennylane as qml
from pennylane import numpy as pnp

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)

## Experiment 1: Initial VQC

EXPERIMENTS:

Layers: 
- Trainable rotations: Rot has three trainable angles.
- Entanglement: CNOT

| Qubits | Layers | Parameters |
| -----: | -----: | ---------: |
|      4 |      2 |         24 |
|      4 |      4 |         48 |
|      4 |      6 |         72 |
|      8 |      2 |         48 |
|      8 |      4 |         96 |
|      8 |      6 |        144 |
|     16 |      2 |         96 |
|     16 |      4 |        192 |
|     16 |      6 |        288 |

- EPOCHS
- BATCH SIZE
- LEARNING RATE


EXPERIMENT 1 SETTINGS:

- 4, 8, 12, 16 qubits/PCA
- N_LAYERS = 2
- N_EPOCHS = 30
- LEARNING_RATE = 0.05
- BATCH_SIZE = 32
- RANDOM_SEED = 42

In [ ]:
# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent.parent

ANGLE_ROOT = Path(PROJECT_ROOT/"data/vqc/angle_encoding")

RESULTS_ROOT = Path(PROJECT_ROOT/"results/vqc/experiment3")

RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

EXCEL_PATH = (
    RESULTS_ROOT /
    "vqc_pca_qubit_comparison.xlsx"
)

# ============================================================
# SETTINGS
# ============================================================

SUBJECTS = [
    "002",
    "005",
    "006",
    "093"
]

QUBIT_LIST = [
    4,
    8,
    12,
    16
]

N_LAYERS = 2
N_CLASSES = 4
N_EPOCHS = 30
LEARNING_RATE = 0.05
BATCH_SIZE = 32
RANDOM_SEED = 42

TASK_NAMES = [
    "auditory",
    "somatosensory",
    "motor",
    "rest"
]

# RANDOM SEED
np.random.seed(
    RANDOM_SEED
)

# ============================================================
# FUNCTIONS
# ============================================================

def softmax(x):

    x = np.asarray(x)

    x = x - np.max(
        x,
        axis=-1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return (
        exp_x
        /
        np.sum(
            exp_x,
            axis=-1,
            keepdims=True
        )
    )

def initialise_weights(n_qubits, seed):

    rng = np.random.default_rng(seed)

    weights = (
        0.01
        * rng.standard_normal(
            (
                N_LAYERS,
                n_qubits,
                3
            )
        )
    )

    return pnp.array(
        weights,
        requires_grad=True
    )

def create_quantum_circuit(n_qubits):

    dev = qml.device(
        "default.qubit",
        wires=n_qubits
    )

    @qml.qnode(dev)
    def quantum_circuit(
        x,
        weights
    ):

        # ANGLE ENCODING
        qml.AngleEmbedding(
            x,
            wires=range(n_qubits),
            rotation="Y"
        )

        for layer in range(N_LAYERS):

            # TRAINABLE ROTATIONS
            for qubit in range(n_qubits):

                # Each Rot gate has three trainable parameters.
                # So 2 layers × 8 qubits × 3 = 48 trainable parameters.
                qml.Rot(
                    weights[layer, qubit, 0],
                    weights[layer, qubit, 1],
                    weights[layer, qubit, 2],
                    wires=qubit
                )

            # ENTANGLEMENT
            for qubit in range(
                n_qubits - 1
            ):

                qml.CNOT(
                    wires=[
                        qubit,
                        qubit + 1
                    ]
                )

        # MEASUREMENTS
        return [
            qml.expval(
                qml.PauliZ(i)
            )
            for i in range(N_CLASSES)
        ]

    return quantum_circuit

# FORWARD PASS

def predict_logits(
    X,
    weights,
    quantum_circuit
):

    outputs = []

    for sample in X:

        result = quantum_circuit(
            sample,
            weights
        )

        outputs.append(
            np.asarray(result)
        )

    return pnp.asarray(
        outputs
    )


# CROSS-ENTROPY

def cross_entropy(
    probabilities,
    labels
):

    probabilities = pnp.clip(
        probabilities,
        1e-10,
        1.0
    )

    losses = -pnp.log(
        probabilities[
            np.arange(
                len(labels)
            ),
            labels
        ]
    )

    return pnp.mean(
        losses
    )

# ============================================================
# TRAIN VQC
# ============================================================

def train_vqc(
    X_train,
    y_train,
    n_qubits,
    seed
):

    quantum_circuit = (
        create_quantum_circuit(
            n_qubits
        )
    )

    weights = initialise_weights(n_qubits, seed)

    # OPTIMISER
    opt = qml.AdamOptimizer(
        stepsize=LEARNING_RATE
    )

    loss_history = []
    accuracy_history = []

    # Reproducible batch generator
    rng = np.random.default_rng(seed)

    # COST FUNCTION
    def cost_fn(
        weights,
        X_batch,
        y_batch
    ):

        logits = predict_logits(
            X_batch,
            weights,
            quantum_circuit
        )

        probabilities = softmax(
            logits
        )

        return cross_entropy(
            probabilities,
            y_batch
        )

    # TRAINING LOOP

    for epoch in range(
        N_EPOCHS
    ):

        # Random mini-batch
        batch_size = min(
            BATCH_SIZE,
            len(X_train)
        )

        batch_indices = rng.choice(
            len(X_train),
            size=batch_size,
            replace=False
        )

        X_batch = X_train[
            batch_indices
        ]

        y_batch = y_train[
            batch_indices
        ]

        # Update weights
        weights, loss = opt.step_and_cost(
            lambda w:
                cost_fn(
                    w,
                    X_batch,
                    y_batch
                ),
            weights
        )

        loss_history.append(
            float(loss)
        )

        # Training accuracy
        # ----------------------------------------------------

        train_predictions, _ = predict(
            X_batch,
            weights,
            quantum_circuit
        )

        batch_accuracy = accuracy_score(
            y_batch,
            train_predictions
        )

        accuracy_history.append(
            batch_accuracy
        )

        if (
            epoch == 0
            or
            (epoch + 1) % 5 == 0
        ):

            print(
                f"Epoch "
                f"{epoch + 1:3d}/{N_EPOCHS} "
                f"| Loss = "
                f"{loss:.6f}"
            )

    return (
        weights,
        quantum_circuit,
        loss_history,
        accuracy_history
    )

# ============================================================
# PREDICTION
# ============================================================

def predict(
    X,
    weights,
    quantum_circuit
):

    logits = predict_logits(
        X,
        weights,
        quantum_circuit
    )

    probabilities = softmax(
        logits
    )

    predictions = pnp.argmax(
        probabilities,
        axis=1
    )

    return (
        np.asarray(predictions),
        np.asarray(probabilities)
    )

all_results = []
task_results = []

for n_qubits in QUBIT_LIST:

    print("\n")
    print("=" * 80)
    print(
        f"VQC EXPERIMENT: "
        f"{n_qubits} QUBITS"
    )
    print("=" * 80)

    # Check angle encoding directory

    for test_subject in SUBJECTS:

        print("\n")
        print("-" * 80)

        print(
            f"QUBITS = {n_qubits}"
        )

        print(
            f"TEST SUBJECT = "
            f"{test_subject}"
        )

        print("-" * 80)

        # Load corresponding angle data

        angle_path = (
            ANGLE_ROOT
            /
            f"loso_test_{test_subject}"
            /
            f"angle_{n_qubits}"
            /
            "data.npz"
        )

        if not angle_path.exists():

            raise FileNotFoundError(
                f"\nMissing angle file:\n"
                f"{angle_path}\n\n"
                f"Make sure PCA/angle encoding "
                f"has been generated for "
                f"{n_qubits} components."
            )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        X_train = data[
            "angles_train"
        ]

        X_test = data[
            "angles_test"
        ]

        y_train = data[
            "y_train"
        ]

        y_test = data[
            "y_test"
        ]

        # Check dimensions

        if X_train.shape[1] != n_qubits:

            raise ValueError(
                f"Expected {n_qubits} "
                f"features but received "
                f"{X_train.shape[1]}"
            )

        print(
            "Training:",
            X_train.shape
        )

        print(
            "Testing:",
            X_test.shape
        )

        print(
            "Training labels:",
            np.unique(y_train)
        )

        print(
            "Testing labels:",
            np.unique(y_test)
        )

        # TRAIN

        print("\n")
        print(
            "Training VQC..."
        )

        (
            weights,
            quantum_circuit,
            loss_history,
            accuracy_history
        ) = train_vqc(
            X_train,
            y_train,
            n_qubits,
            seed=RANDOM_SEED
            + n_qubits * 100
            + int(test_subject)
        )

        # TEST

        print("\n")
        print(
            "Testing VQC..."
        )

        predictions, probabilities = (
            predict(
                X_test,
                weights,
                quantum_circuit
            )
        )

        tasks_test = data["tasks_test"]

        # METRICS

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        balanced_accuracy = (
            balanced_accuracy_score(
                y_test,
                predictions
            )
        )

        macro_f1 = f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_test,
            predictions,
            average="weighted",
            zero_division=0
        )

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=[
                0,
                1,
                2,
                3
            ]
        )

        print(
            "\nTest accuracy:",
            f"{accuracy:.4f}"
        )

        print(
            "Balanced accuracy:",
            f"{balanced_accuracy:.4f}"
        )

        print(
            "Macro F1:",
            f"{macro_f1:.4f}"
        )

        print(
            "Weighted F1:",
            f"{weighted_f1:.4f}"
        )

        print(
            "\nConfusion matrix:"
        )

        print(cm)

        # SAVE INDIVIDUAL RESULT

        result_dir = (
            RESULTS_ROOT
            /
            f"qubits_{n_qubits}"
        )

        result_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        save_path = (
            result_dir
            /
            f"loso_test_{test_subject}.npz"
        )

        np.savez_compressed(

            save_path,
            test_subject=test_subject,
            predictions=predictions,
            probabilities=probabilities,
            y_test=y_test,
            accuracy=accuracy,
            balanced_accuracy=balanced_accuracy,
            macro_f1=macro_f1,
            weighted_f1=weighted_f1,
            confusion_matrix=cm,
            weights=np.asarray(weights),
            loss_history=np.asarray(
                loss_history
            ),
            accuracy_history=np.asarray(
                accuracy_history
            ),
            n_qubits=n_qubits,
            n_layers=N_LAYERS
        )

        print(
            "Saved:",
            save_path
        )

        # STORE ROW FOR EXCEL

        all_results.append({

            "subject": test_subject,
            "n_qubits": n_qubits,
            "accuracy": accuracy,
            "balanced_accuracy":
                balanced_accuracy,
            "macro_f1":
                macro_f1,
            "weighted_f1":
                weighted_f1,
            "final_training_loss":
                loss_history[-1]
        })

        for class_id, task_name in enumerate(TASK_NAMES):

            task_mask = (
                y_test == class_id
            )

            n_samples = int(
                np.sum(task_mask)
            )

            if n_samples == 0:
                continue

            task_precision = precision_score(
                y_test,
                predictions,
                labels=[class_id],
                average="macro",
                zero_division=0
            )

            task_recall = recall_score(
                y_test,
                predictions,
                labels=[class_id],
                average="macro",
                zero_division=0
            )

            task_f1 = f1_score(
                y_test,
                predictions,
                labels=[class_id],
                average="macro",
                zero_division=0
            )

            task_results.append({

                "subject":
                    test_subject,

                "n_qubits":
                    n_qubits,

                "task":
                    task_name,

                "n_test_samples":
                    n_samples,

                "precision":
                    task_precision,

                "recall":
                    task_recall,

                "f1":
                    task_f1
            })

# CONVERT RESULTS TO DATAFRAME

results_df = pd.DataFrame(
    all_results
)

task_results_df = pd.DataFrame(
    task_results
)
    
# LOSO SUMMARY

summary_df = (
    results_df
    .groupby(
        "n_qubits"
    )
    .agg({

        "accuracy":
            ["mean", "std"],

        "balanced_accuracy":
            ["mean", "std"],

        "macro_f1":
            ["mean", "std"],

        "weighted_f1":
            ["mean", "std"],

        "final_training_loss":
            ["mean", "std"]

    })
    .reset_index()
)

# Flatten column names

summary_df.columns = [

    "n_qubits",

    "accuracy_mean",
    "accuracy_std",

    "balanced_accuracy_mean",
    "balanced_accuracy_std",

    "macro_f1_mean",
    "macro_f1_std",

    "weighted_f1_mean",
    "weighted_f1_std",

    "loss_mean",
    "loss_std"

]

# SAVE EXCEL

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="LOSO Results",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    task_results_df.to_excel(
        writer,
        sheet_name="Task Results",
        index=False
    )


print("\n")
print("=" * 80)
print("EXCEL RESULTS SAVED")
print("=" * 80)

print(
    EXCEL_PATH
)

# PRINT SUMMARY

print("\n")
print("=" * 80)
print("VQC QUANTUM DIMENSION SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False
    )
)

# ============================================================
# PLOT 1:
# MEAN ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["accuracy_mean"],
    yerr=summary_df["accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "VQC Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


accuracy_plot = (
    RESULTS_ROOT /
    "accuracy_vs_qubits.png"
)

plt.savefig(
    accuracy_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 2:
# MACRO F1 VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["macro_f1_mean"],
    yerr=summary_df["macro_f1_std"],
    marker="o",
    capsize=5
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Macro F1"
)

plt.title(
    "VQC Macro F1 vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()


f1_plot = (
    RESULTS_ROOT /
    "macro_f1_vs_qubits.png"
)

plt.savefig(
    f1_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 3:
# BALANCED ACCURACY VS NUMBER OF QUBITS
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.errorbar(
    summary_df["n_qubits"],
    summary_df["balanced_accuracy_mean"],
    yerr=summary_df["balanced_accuracy_std"],
    marker="o",
    capsize=5
)

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Balanced accuracy"
)

plt.title(
    "VQC Balanced Accuracy vs Number of Qubits"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


balanced_plot = (
    RESULTS_ROOT /
    "balanced_accuracy_vs_qubits.png"
)

plt.savefig(
    balanced_plot,
    dpi=300
)

plt.close()


# ============================================================
# PLOT 4:
# ACCURACY FOR EACH SUBJECT
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = results_df[
        results_df["subject"] == subject
    ]

    plt.plot(
        subject_data["n_qubits"],
        subject_data["accuracy"],
        marker="o",
        label=f"Subject {subject}"
    )


plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "LOSO accuracy"
)

plt.title(
    "LOSO Accuracy Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()


subject_plot = (
    RESULTS_ROOT /
    "accuracy_by_subject.png"
)

plt.savefig(
    subject_plot,
    dpi=300
)

plt.close()

# ============================================================
# PLOT 5:
# NUMBER OF TRAINING SAMPLES PER TASK
# ============================================================

task_counts = []

for test_subject in SUBJECTS:

    for n_qubits in QUBIT_LIST:

        angle_path = (
            ANGLE_ROOT
            / f"loso_test_{test_subject}"
            / f"angle_{n_qubits}"
            / "data.npz"
        )

        data = np.load(
            angle_path,
            allow_pickle=True
        )

        y_train = data["y_train"]

        for class_id, task_name in enumerate(TASK_NAMES):

            count = np.sum(
                y_train == class_id
            )

            task_counts.append({
                "subject": test_subject,
                "n_qubits": n_qubits,
                "task": task_name,
                "count": count
            })

task_counts_df = pd.DataFrame(
    task_counts
)

distribution_df = (
    task_counts_df[
        task_counts_df["n_qubits"] == 4
    ]
)

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = distribution_df[
        distribution_df["subject"] == subject
    ]

    plt.plot(
        subject_data["task"],
        subject_data["count"],
        marker="o",
        label=f"Subject {subject}"
    )

plt.xlabel("Task")
plt.ylabel("Number of training trials")
plt.title(
    "Training Trial Distribution Across Tasks"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

task_distribution_plot = (
    RESULTS_ROOT /
    "training_samples_by_task.png"
)

plt.savefig(
    task_distribution_plot,
    dpi=300
)

plt.close()

# ============================================================
# PLOT 6:
# Training Data Composition Across Tasks
# ============================================================

task_percentage = (
    distribution_df
    .groupby(["subject", "task"])["count"]
    .sum()
    .reset_index()
)

task_percentage["percentage"] = (
    task_percentage["count"]
    /
    task_percentage.groupby("subject")["count"]
        .transform("sum")
    * 100
)

plt.figure(
    figsize=(9, 6)
)

for subject in SUBJECTS:

    subject_data = task_percentage[
        task_percentage["subject"] == subject
    ]

    plt.plot(
        subject_data["task"],
        subject_data["percentage"],
        marker="o",
        label=f"Subject {subject}"
    )

plt.xlabel("Task")
plt.ylabel("Training samples (%)")
plt.title(
    "Training Data Composition Across Tasks"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "training_task_percentage.png",
    dpi=300
)

plt.close()

# ============================================================
# PLOT 7:
# Task-wise F1 Across Quantum Dimensions
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["f1"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["f1"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task F1"
)

plt.title(
    "Task-wise F1 Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_f1_vs_qubits.png",
    dpi=300
)

plt.close()

# ==========================================================
# Task-wise VQC Recall Across Quantum Dimensions
# ==========================================================

plt.figure(
    figsize=(9, 6)
)

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["recall"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["recall"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task recall"
)

plt.title(
    "Task-wise VQC Recall Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_recall_vs_qubits.png",
    dpi=300
)

plt.close()

# ============================================================
# TASK PRECISION
# ============================================================

plt.figure(figsize=(9, 6))

for task in TASK_NAMES:

    task_data = (
        task_results_df[
            task_results_df["task"] == task
        ]
        .groupby("n_qubits")["precision"]
        .mean()
        .reset_index()
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["precision"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean task precision"
)

plt.title(
    "Task-wise Precision Across Quantum Dimensions"
)

plt.xticks(QUBIT_LIST)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_precision_vs_qubits.png",
    dpi=300
)

plt.close()

# ============================================================
# TRAINING LOSS CURVE
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    range(1, len(loss_history) + 1),
    loss_history,
    marker="o"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training cross-entropy"
)

plt.title(
    f"VQC Training Loss — "
    f"{n_qubits} Qubits — "
    f"Test Subject {test_subject}"
)

plt.grid(alpha=0.3)
plt.tight_layout()

loss_plot = (
    RESULTS_ROOT
    / f"loss_q{n_qubits}_subject_{test_subject}.png"
)

plt.savefig(
    loss_plot,
    dpi=300
)

plt.close()

# ============================================================
# CONFUSION MATRIX
# ============================================================

plt.figure(
    figsize=(7, 6)
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=TASK_NAMES,
    yticklabels=TASK_NAMES
)

plt.xlabel(
    "Predicted task"
)

plt.ylabel(
    "True task"
)

plt.title(
    f"Confusion Matrix — "
    f"{n_qubits} Qubits — "
    f"Subject {test_subject}"
)

plt.tight_layout()

cm_plot = (
    RESULTS_ROOT
    / f"confusion_q{n_qubits}_subject_{test_subject}.png"
)

plt.savefig(
    cm_plot,
    dpi=300
)

plt.close()

# ============================================================
# COMPLETE
# ============================================================

print("\n")
print("=" * 80)
print("ALL VQC EXPERIMENTS COMPLETE")
print("=" * 80)

print(
    "Excel:",
    EXCEL_PATH
)

print(
    "Plots saved in:",
    RESULTS_ROOT
)



VQC EXPERIMENT: 4 QUBITS


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 002
--------------------------------------------------------------------------------
Training: (4369, 4)
Testing: (1484, 4)
Training labels: [0 1 2 3]
Testing labels: [0 1 2 3]


Training VQC...
Epoch   1/30 | Loss = 1.527388
Epoch   5/30 | Loss = 1.439158
Epoch  10/30 | Loss = 1.394878
Epoch  15/30 | Loss = 1.356183
Epoch  20/30 | Loss = 1.354772
Epoch  25/30 | Loss = 1.275262
Epoch  30/30 | Loss = 1.422201


Testing VQC...

Test accuracy: 0.3120
Balanced accuracy: 0.2525
Macro F1: 0.1632
Weighted F1: 0.1877

Confusion matrix:
[[ 22   2   0 376]
 [ 33   1   0 371]
 [ 28   1   0 188]
 [ 21   1   0 440]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment3/qubits_4/loso_test_002.npz


--------------------------------------------------------------------------------
QUBITS = 4
TEST SUBJECT = 005
--------------------------------------------

In [14]:
# ============================================================
# NORMALISED OVERALL CONFUSION MATRICES
# ============================================================

for n_qubits in QUBIT_LIST:

    cm = confusion_matrices[n_qubits]

    row_sums = cm.sum(
        axis=1,
        keepdims=True
    )

    cm_normalised = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(
            cm,
            dtype=float
        ),
        where=row_sums != 0
    )

    plt.figure(
        figsize=(7, 6)
    )

    sns.heatmap(
        cm_normalised,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=TASK_NAMES,
        yticklabels=TASK_NAMES,
        vmin=0,
        vmax=1
    )

    plt.xlabel(
        "Predicted task"
    )

    plt.ylabel(
        "True task"
    )

    plt.title(
        f"Normalised Overall Confusion Matrix — "
        f"{n_qubits} Qubits"
    )

    plt.tight_layout()

    plt.savefig(
        RESULTS_ROOT /
        f"overall_confusion_normalised_q{n_qubits}.png",
        dpi=300
    )

    plt.close()

In [15]:
# ============================================================
# OVERALL TRAINING LOSS
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for n_qubits in QUBIT_LIST:

    histories = loss_histories[n_qubits]

    if len(histories) == 0:
        continue

    histories = np.asarray(
        histories
    )

    mean_loss = np.mean(
        histories,
        axis=0
    )

    std_loss = np.std(
        histories,
        axis=0
    )

    epochs = np.arange(
        1,
        len(mean_loss) + 1
    )

    plt.plot(
        epochs,
        mean_loss,
        marker="o",
        label=f"{n_qubits} qubits"
    )

    plt.fill_between(
        epochs,
        mean_loss - std_loss,
        mean_loss + std_loss,
        alpha=0.15
    )

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training cross-entropy"
)

plt.title(
    "Overall VQC Training Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_loss.png",
    dpi=300
)

plt.close()

In [16]:
# ============================================================
# OVERALL TRAINING ACCURACY
# ============================================================

plt.figure(
    figsize=(9, 6)
)

for n_qubits in QUBIT_LIST:

    histories = accuracy_histories[n_qubits]

    if len(histories) == 0:
        continue

    histories = np.asarray(
        histories
    )

    mean_accuracy = np.mean(
        histories,
        axis=0
    )

    std_accuracy = np.std(
        histories,
        axis=0
    )

    epochs = np.arange(
        1,
        len(mean_accuracy) + 1
    )

    plt.plot(
        epochs,
        mean_accuracy,
        marker="o",
        label=f"{n_qubits} qubits"
    )

    plt.fill_between(
        epochs,
        mean_accuracy - std_accuracy,
        mean_accuracy + std_accuracy,
        alpha=0.15
    )

plt.axhline(
    0.25,
    linestyle="--",
    label="Random baseline (25%)"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Training accuracy"
)

plt.title(
    "Overall VQC Training Accuracy"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_accuracy.png",
    dpi=300
)

plt.close()

In [17]:
# ============================================================
# TASK F1 HEATMAP
# ============================================================

task_f1_summary = (
    task_results_df
    .groupby(
        ["task", "n_qubits"]
    )["f1"]
    .mean()
    .reset_index()
)

task_f1_pivot = (
    task_f1_summary
    .pivot(
        index="task",
        columns="n_qubits",
        values="f1"
    )
)

plt.figure(
    figsize=(9, 5)
)

sns.heatmap(
    task_f1_pivot,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    vmin=0,
    vmax=1
)

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Task"
)

plt.title(
    "Mean Task-wise F1 Across Quantum Dimensions"
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "task_f1_heatmap.png",
    dpi=300
)

plt.close()

In [18]:
# ============================================================
# OVERALL TRAINING SAMPLE DISTRIBUTION
# ============================================================

overall_task_counts = (
    task_counts_df[
        task_counts_df["n_qubits"] == 4
    ]
    .groupby("task")["count"]
    .mean()
    .reindex(TASK_NAMES)
)

plt.figure(
    figsize=(9, 6)
)

plt.bar(
    overall_task_counts.index,
    overall_task_counts.values
)

plt.xlabel(
    "Task"
)

plt.ylabel(
    "Mean number of training trials"
)

plt.title(
    "Training Trial Distribution Across Tasks"
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "overall_training_task_distribution.png",
    dpi=300
)

plt.close()

In [19]:
# ============================================================
# PREDICTED CLASS DISTRIBUTION
# ============================================================

prediction_distribution = []

for n_qubits in QUBIT_LIST:

    for test_subject in SUBJECTS:

        result_path = (
            RESULTS_ROOT
            / f"qubits_{n_qubits}"
            / f"loso_test_{test_subject}.npz"
        )

        if not result_path.exists():
            continue

        result = np.load(
            result_path,
            allow_pickle=True
        )

        predictions = result[
            "predictions"
        ]

        for class_id, task_name in enumerate(TASK_NAMES):

            count = np.sum(
                predictions == class_id
            )

            prediction_distribution.append({

                "n_qubits":
                    n_qubits,

                "subject":
                    test_subject,

                "task":
                    task_name,

                "count":
                    count
            })

prediction_distribution_df = pd.DataFrame(
    prediction_distribution
)

prediction_summary = (
    prediction_distribution_df
    .groupby(
        ["n_qubits", "task"]
    )["count"]
    .mean()
    .reset_index()
)

plt.figure(
    figsize=(10, 6)
)

for task in TASK_NAMES:

    task_data = (
        prediction_summary[
            prediction_summary["task"] == task
        ]
    )

    plt.plot(
        task_data["n_qubits"],
        task_data["count"],
        marker="o",
        label=task
    )

plt.xlabel(
    "Number of qubits / PCA components"
)

plt.ylabel(
    "Mean number of predictions"
)

plt.title(
    "Predicted Task Distribution Across Quantum Dimensions"
)

plt.xticks(
    QUBIT_LIST
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT /
    "predicted_task_distribution.png",
    dpi=300
)

plt.close()

In [20]:
# ============================================================
# OVERALL CONFUSION MATRICES
# POOLED ACROSS ALL LOSO SUBJECTS
# ============================================================

for n_qubits in QUBIT_LIST:

    all_y_true = []
    all_y_pred = []

    for test_subject in SUBJECTS:

        result_path = (
            RESULTS_ROOT
            / f"qubits_{n_qubits}"
            / f"loso_test_{test_subject}.npz"
        )

        if not result_path.exists():

            print(
                f"Missing result: {result_path}"
            )

            continue

        data = np.load(
            result_path,
            allow_pickle=True
        )

        y_true = data["y_test"]
        y_pred = data["predictions"]

        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)

    # Convert to arrays

    all_y_true = np.asarray(
        all_y_true
    )

    all_y_pred = np.asarray(
        all_y_pred
    )

    # Overall confusion matrix

    overall_cm = confusion_matrix(
        all_y_true,
        all_y_pred,
        labels=[0, 1, 2, 3]
    )

    print("\n")
    print("=" * 70)
    print(
        f"OVERALL CONFUSION MATRIX — "
        f"{n_qubits} QUBITS"
    )
    print("=" * 70)

    print(overall_cm)

    # --------------------------------------------------------
    # PLOT
    # --------------------------------------------------------

    plt.figure(
        figsize=(7, 6)
    )

    sns.heatmap(
        overall_cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=TASK_NAMES,
        yticklabels=TASK_NAMES,
        cbar_kws={
            "label": "Number of test trials"
        }
    )

    plt.xlabel(
        "Predicted task"
    )

    plt.ylabel(
        "True task"
    )

    plt.title(
        f"Overall Confusion Matrix — "
        f"{n_qubits} Qubits"
    )

    plt.tight_layout()

    cm_plot = (
        RESULTS_ROOT
        / f"overall_confusion_q{n_qubits}.png"
    )

    plt.savefig(
        cm_plot,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        "Saved:",
        cm_plot
    )



OVERALL CONFUSION MATRIX — 4 QUBITS
[[273 630   0 697]
 [298 673   0 658]
 [160 345   0 345]
 [280 728   2 764]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment3/overall_confusion_q4.png


OVERALL CONFUSION MATRIX — 8 QUBITS
[[ 205  248    0 1147]
 [ 268  287    1 1073]
 [ 123   84    0  643]
 [ 226  252    2 1294]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment3/overall_confusion_q8.png


OVERALL CONFUSION MATRIX — 12 QUBITS
[[1118  140    0  342]
 [1163  176    1  289]
 [ 545   81    0  224]
 [1341   37    2  394]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment3/overall_confusion_q12.png


OVERALL CONFUSION MATRIX — 16 QUBITS
[[1033  100    0  467]
 [ 964  129    0  536]
 [ 496   59    0  295]
 [1121   98    2  553]]
Saved: /home/master/MasterThesis/OPM-MEG MPS/results/vqc/experiment3/overall_confusion_q16.png


## Test VQC

VQC TRAINABILITY TEST

In [ ]:
N_QUBITS = 8
N_LAYERS = 2

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)


@qml.qnode(
    dev,
    interface="autograd"
)
def circuit(x, weights):

    # -----------------------------
    # Angle encoding
    # -----------------------------

    for q in range(N_QUBITS):

        qml.RY(
            x[q],
            wires=q
        )

    # -----------------------------
    # Trainable layers
    # -----------------------------

    for layer in range(N_LAYERS):

        for q in range(N_QUBITS):

            qml.RY(
                weights[layer, q, 0],
                wires=q
            )

            qml.RZ(
                weights[layer, q, 1],
                wires=q
            )

        # Entanglement
        for q in range(N_QUBITS - 1):

            qml.CNOT(
                wires=[
                    q,
                    q + 1
                ]
            )

    return [
        qml.expval(
            qml.PauliZ(q)
        )
        for q in range(N_QUBITS)
    ]


# ============================================================
# CREATE TRAINABLE PARAMETERS
# ============================================================

weights = np.array(
    0.01 * np.random.randn(
        N_LAYERS,
        N_QUBITS,
        2
    ),
    requires_grad=True
)

x = np.array(
    np.random.uniform(
        0,
        np.pi,
        N_QUBITS
    )
)

print("=" * 70)
print("VQC TRAINABILITY TEST")
print("=" * 70)

print(
    "Weights shape:",
    weights.shape
)

print(
    "Weights requires grad:",
    weights.requires_grad
)


# ============================================================
# CIRCUIT OUTPUT
# ============================================================

output = circuit(
    x,
    weights
)

print(
    "\nCircuit output:"
)

print(output)


# ============================================================
# SIMPLE LOSS
# ============================================================

def loss_fn(weights):

    output = circuit(
        x,
        weights
    )

    return qml.math.sum(
        qml.math.stack(output)
    )


loss_before = loss_fn(weights)

print(
    "\nLoss before:",
    loss_before
)


# ============================================================
# GRADIENT
# ============================================================

gradient = qml.grad(
    loss_fn
)(weights)

print(
    "\nGradient shape:",
    gradient.shape
)

print(
    "Gradient norm:",
    np.linalg.norm(gradient)
)


# ============================================================
# TEST OPTIMIZER
# ============================================================

opt = qml.AdamOptimizer(
    stepsize=0.01
)

weights_new, loss_new = opt.step_and_cost(
    loss_fn,
    weights
)

print(
    "\nLoss after one optimizer step:",
    loss_new
)

print(
    "Weight change:",
    np.linalg.norm(
        weights_new - weights
    )
)


if np.linalg.norm(gradient) > 1e-10:

    print(
        "\nPASS: VQC has non-zero gradients."
    )

else:

    print(
        "\nFAIL: VQC gradient is zero."
    )


if np.linalg.norm(
    weights_new - weights
) > 1e-10:

    print(
        "PASS: optimizer changed "
        "the VQC parameters."
    )

else:

    print(
        "FAIL: optimizer did not "
        "change the parameters."
    )

VQC TRAINABILITY TEST
Weights shape: (2, 8, 2)
Weights requires grad: True

Circuit output:
[tensor(-0.99711141, requires_grad=True), tensor(0.03261246, requires_grad=True), tensor(0.95321531, requires_grad=True), tensor(0.01022093, requires_grad=True), tensor(0.43151944, requires_grad=True), tensor(-0.00596899, requires_grad=True), tensor(-0.22071696, requires_grad=True), tensor(0.00589842, requires_grad=True)]

Loss before: 0.20966919547780882

Gradient shape: (2, 8, 2)
Gradient norm: 1.6291960122612252

Loss after one optimizer step: 0.20966919547780882
Weight change: 0.04859651462308379

PASS: VQC has non-zero gradients.
PASS: optimizer changed the VQC parameters.


Test 1 - Class Balance

In [23]:
# ============================================================
# TEST 1: CLASS DISTRIBUTION
# ============================================================

print("="*80)
print("CLASS DISTRIBUTION TEST")
print("="*80)

for subject in SUBJECTS:

    path = (
        ANGLE_ROOT
        /
        f"loso_test_{subject}"
        /
        "angle_8"
        /
        "data.npz"
    )

    data = np.load(path)

    y_train = data["y_train"]
    y_test = data["y_test"]

    print("\nSubject:", subject)

    print("Training:")
    unique, counts = np.unique(
        y_train,
        return_counts=True
    )

    for u,c in zip(unique,counts):
        print(
            f" Class {u}: {c} "
            f"({100*c/len(y_train):.2f}%)"
        )


    print("Testing:")

    unique, counts = np.unique(
        y_test,
        return_counts=True
    )

    for u,c in zip(unique,counts):
        print(
            f" Class {u}: {c} "
            f"({100*c/len(y_test):.2f}%)"
        )

CLASS DISTRIBUTION TEST

Subject: 002
Training:
 Class 0: 1200 (27.47%)
 Class 1: 1224 (28.02%)
 Class 2: 633 (14.49%)
 Class 3: 1312 (30.03%)
Testing:
 Class 0: 400 (26.95%)
 Class 1: 405 (27.29%)
 Class 2: 217 (14.62%)
 Class 3: 462 (31.13%)

Subject: 005
Training:
 Class 0: 1200 (27.47%)
 Class 1: 1229 (28.13%)
 Class 2: 604 (13.82%)
 Class 3: 1336 (30.58%)
Testing:
 Class 0: 400 (26.95%)
 Class 1: 400 (26.95%)
 Class 2: 246 (16.58%)
 Class 3: 438 (29.51%)

Subject: 006
Training:
 Class 0: 1200 (27.01%)
 Class 1: 1222 (27.51%)
 Class 2: 684 (15.40%)
 Class 3: 1336 (30.08%)
Testing:
 Class 0: 400 (28.35%)
 Class 1: 407 (28.84%)
 Class 2: 166 (11.76%)
 Class 3: 438 (31.04%)

Subject: 093
Training:
 Class 0: 1200 (27.40%)
 Class 1: 1212 (27.68%)
 Class 2: 629 (14.36%)
 Class 3: 1338 (30.55%)
Testing:
 Class 0: 400 (27.14%)
 Class 1: 417 (28.29%)
 Class 2: 221 (14.99%)
 Class 3: 436 (29.58%)


Test 2 - VQC Outputs

In [24]:
# ============================================================
# TEST 2: PREDICTION COLLAPSE
# ============================================================

print("="*80)
print("PREDICTION COLLAPSE TEST")
print("="*80)


for q in QUBIT_LIST:

    for subject in SUBJECTS:


        path = (
            RESULTS_ROOT
            /
            f"qubits_{q}"
            /
            f"loso_test_{subject}.npz"
        )


        data = np.load(path)

        predictions = data["predictions"]


        unique, counts = np.unique(
            predictions,
            return_counts=True
        )


        print(
            f"\nQubits {q} | Subject {subject}"
        )

        for u,c in zip(unique,counts):

            print(
                f"Prediction {u}: "
                f"{c} "
                f"({100*c/len(predictions):.2f}%)"
            )

PREDICTION COLLAPSE TEST

Qubits 4 | Subject 002
Prediction 0: 19 (1.28%)
Prediction 1: 1458 (98.25%)
Prediction 3: 7 (0.47%)

Qubits 4 | Subject 005
Prediction 0: 3 (0.20%)
Prediction 1: 1356 (91.37%)
Prediction 2: 1 (0.07%)
Prediction 3: 124 (8.36%)

Qubits 4 | Subject 006
Prediction 0: 3 (0.21%)
Prediction 1: 77 (5.46%)
Prediction 2: 2 (0.14%)
Prediction 3: 1329 (94.19%)

Qubits 4 | Subject 093
Prediction 0: 602 (40.84%)
Prediction 1: 667 (45.25%)
Prediction 2: 6 (0.41%)
Prediction 3: 199 (13.50%)

Qubits 8 | Subject 002
Prediction 0: 1480 (99.73%)
Prediction 1: 2 (0.13%)
Prediction 3: 2 (0.13%)

Qubits 8 | Subject 005
Prediction 0: 96 (6.47%)
Prediction 1: 117 (7.88%)
Prediction 2: 1 (0.07%)
Prediction 3: 1270 (85.58%)

Qubits 8 | Subject 006
Prediction 0: 226 (16.02%)
Prediction 1: 203 (14.39%)
Prediction 2: 2 (0.14%)
Prediction 3: 980 (69.45%)

Qubits 8 | Subject 093
Prediction 0: 652 (44.23%)
Prediction 1: 387 (26.26%)
Prediction 2: 24 (1.63%)
Prediction 3: 411 (27.88%)

Qubits 

Test 3

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score


print("="*80)
print("CLASSICAL BASELINE")
print("="*80)


for subject in SUBJECTS:


    path = (
        ANGLE_ROOT
        /
        f"loso_test_{subject}"
        /
        "angle_8"
        /
        "data.npz"
    )


    data = np.load(path)


    X_train=data["angles_train"]
    X_test=data["angles_test"]

    y_train=data["y_train"]
    y_test=data["y_test"]


    clf = LogisticRegression(
        max_iter=1000
    )


    clf.fit(
        X_train,
        y_train
    )


    pred = clf.predict(
        X_test
    )


    acc = accuracy_score(
        y_test,
        pred
    )


    f1 = f1_score(
        y_test,
        pred,
        average="macro"
    )


    print(
        subject,
        "Accuracy:",
        acc,
        "Macro F1:",
        f1
    )

CLASSICAL BASELINE
002 Accuracy: 0.316711590296496 Macro F1: 0.18241756382154023
005 Accuracy: 0.3106469002695418 Macro F1: 0.21021917053980643
006 Accuracy: 0.293408929836995 Macro F1: 0.2064100414770636
093 Accuracy: 0.3331071913161465 Macro F1: 0.24528845037478747


Test 4 - Overfit

In [ ]:
# ============================================================
# TEST 4: OVERFIT TEST
# ============================================================


subject="002"
q=8


path=(
    ANGLE_ROOT
    /
    f"loso_test_{subject}"
    /
    f"angle_{q}"
    /
    "data.npz"
)


data=np.load(path)


X=data["angles_train"]
y=data["y_train"]


weights,circuit,loss=train_vqc(
    X,
    y,
    q
)


pred,_=predict(
    X,
    weights,
    circuit
)


train_acc=accuracy_score(
    y,
    pred
)


print(
    "Training accuracy:",
    train_acc
)

Epoch   1/30 | Loss = 1.430320
Epoch   5/30 | Loss = 1.432528
Epoch  10/30 | Loss = 1.319325
Epoch  15/30 | Loss = 1.360122
Epoch  20/30 | Loss = 1.347962
Epoch  25/30 | Loss = 1.380705
Epoch  30/30 | Loss = 1.335456
Training accuracy: 0.2993820096131838


In [ ]:
# ============================================================
# TEST 5: WEIGHT CHANGE
# ============================================================


initial = initialise_weights(8)


trained, circuit, loss = train_vqc(
    X_train,
    y_train,
    8
)


difference = np.linalg.norm(
    trained-initial
)


print(
    "Weight change:",
    difference
)

Epoch   1/30 | Loss = 1.384204
Epoch   5/30 | Loss = 1.474832
Epoch  10/30 | Loss = 1.343338
Epoch  15/30 | Loss = 1.387859
Epoch  20/30 | Loss = 1.416469
Epoch  25/30 | Loss = 1.390327
Epoch  30/30 | Loss = 1.346557
Weight change: 2.194158132404436


In [9]:
for n_qubits in [4,8,12,16]:

    print("\n")
    print("="*50)
    print("Testing", n_qubits, "qubits")

    circuit = make_vqc(
        n_qubits,
        N_LAYERS
    )

    weights = qml.numpy.array(
        np.random.normal(
            0,
            0.05,
            size=(
                N_LAYERS,
                n_qubits,
                2
            )
        ),
        requires_grad=True
    )


    angles = np.random.uniform(
        0,
        np.pi,
        size=n_qubits
    )


    print(
        "Weights:",
        weights.shape
    )

    print(
        "Angles:",
        angles.shape
    )


    output = circuit(
        angles,
        weights
    )


    print(
        "Output:",
        len(output)
    )



Testing 4 qubits
Weights: (2, 4, 2)
Angles: (4,)
Output: 4


Testing 8 qubits
Weights: (2, 8, 2)
Angles: (8,)
Output: 8


Testing 12 qubits
Weights: (2, 12, 2)
Angles: (12,)
Output: 12


Testing 16 qubits
Weights: (2, 16, 2)
Angles: (16,)
Output: 16
